# MusicSage Separator 2D→1D — PyTorch (реверсивная архитектура)

Разделение музыки на стемы (drums / bass / other / vocals) гибридом,
в котором **спектральная (2D) ветка идёт первой, а волновая (1D) — второй**:

1. **2D спектральная ветка (первая)**: STFT микса → log-магнитуда → 2D U-Net
   с BiLSTM-bottleneck → 4 конкурентные маски (softmax по стемам) →
   стемы = ISTFT(mask · STFT(mix)) — с **фазой микса**;
2. **1D WaveUNet-рефайнер (вторая)**: принимает все 4 стема после ISTFT
   (8 каналов = 4 стема × 2 канала стерео), правит waveform напрямую
   (в т.ч. фазу) и прибавляет входные стемы с обучаемым масштабом
   `dummies` (глобальный residual — сеть «чинит», а не рисует с нуля).

Это обратная топология к `separator_pytorch.ipynb` (там 1D первая, 2D вторая,
объединение гейтом): здесь 2D задаёт маски, а 36M-параметровый 1D-сеть
используется как честный пост-процессинг waveform после ISTFT — как в
`refine.py`/`train_refine.py`, но обученная end-to-end вместе с масками.

Пайплайн (обучение и метрики) повторяет `separator_pytorch.ipynb`:
блочная выборка с LRU-кэшем декодирования, Demucs-стайл аугментации,
L1 + мультирезолюционная спектральная L1 + SI-SDR + mixture consistency,
свип (channels, levels) по val loss, инференс чанками с кросфейдом.


In [ ]:
import math
import os
import random
from collections import OrderedDict

import musdb
import numpy as np
import soundfile as sf

import torch
import torch.nn as nn
import torch.nn.functional as F

torch.manual_seed(0)
np.random.seed(0)
random.seed(0)


In [ ]:
# ============================================================
# Фикс бага stempeg: read_stems вызывает warnings.warning() вместо
# warnings.warn() (AttributeError в worker-процессах DataLoader).
# ============================================================
import warnings

if not hasattr(warnings, "warning"):
    warnings.warning = warnings.warn
warnings.filterwarnings("ignore", message="Stems differ in length")
print("stempeg fix applied")


In [ ]:
# ============================================================
# CONFIG
# ============================================================

SAMPLE_RATE = 32000          # даунсэмпл 44.1 -> 32 кГц
CHUNK_SEC = 5                # random crop: из трека (~7 с) вырезается случайный отрезок 5 с
CHUNK_SAMPLES = SAMPLE_RATE * CHUNK_SEC

# 1D WaveUNet-рефайнер: stride-4 свёртки, LEVELS уровней => вход паддится до 4**LEVELS
STRIDE = 4
LEVELS = 5
PAD_LEN = ((CHUNK_SAMPLES + STRIDE ** LEVELS - 1) // STRIDE ** LEVELS) * STRIDE ** LEVELS

EPS = 1e-4

# Корень MUSDB18: здесь лежат папки train/ и test/ с *.stem.mp4
DB_ROOT = "musdb18/wav"

N_SOURCES = 4
SOURCE_NAMES = ["drums", "bass", "other", "vocals"]

# Мультирезолюционный STFT — только для спектрального члена loss
MR_LOSS_CONFIGS = [
    (4096, 2048),
    (2048, 1024),
    (1024, 512),
]

# Первая (спектральная) ветка: STFT микса -> log-магнитуда -> маски
SPEC_NFFT = 4096
SPEC_HOP = 1024

# Mixture consistency: вес члена L1(сумма стемов - микс) в лоссе
W_CONSIST = 1.0

# Аугментации: stem dropout, crosstalk (утечки), шум в микс
AUG_STEM_DROP_P = 0.15    # вероятность обнулить стем в элементе батча
AUG_BLEED_P = 0.         # вероятность утечки другого стема в текущий
AUG_BLEED_G = (0.05, 0.3) # сила утечки (доля чужого стема)
AUG_NOISE_SNR = (20, 40)  # SNR белого шума в миксе, дБ (None — выкл.)

# Параметры 1D-рефайнера (вторая ветка)
WAVE_CHANNELS = 64
WAVE_LEVELS = 5

# ==== Свип моделей: (channels, levels) 1D-рефайнера ====
SWEEP_CONFIGS = [
    (32, 4),   # самый лёгкий (~1-2M)
    (32, 5),   # (~3M)
    (48, 5),   # (~7M)
    (64, 4),   # мелкий, но широкий (~5M)
    (64, 5),   # базовый (~11M)
    (96, 5),   # широкий (~25M)
    (64, 6),   # глубокий (~23M)
]
SWEEP_EPOCHS = 3
SWEEP_LR = 3e-4
SWEEP_DIR = "checkpoints_pytorch/sweep_21d"

DEVICE = (
    "cuda" if torch.cuda.is_available()
    else "mps" if torch.backends.mps.is_available()
    else "cpu"
)
USE_AMP = DEVICE == "cuda"

print("device:", DEVICE, "| amp:", USE_AMP)
print("chunk:", CHUNK_SAMPLES, "samples | padded:", PAD_LEN)


## 1. Подготовка данных

Идентично `separator_pytorch.ipynb`: `musdb.DB` находит `*.stem.mp4`,
`track.stems` — все 5 потоков (mix + 4 стема), блочная выборка чанков
по песням с LRU-кэшем декодирования (одна песня = один декод за эпоху).


In [ ]:
def _stems_dict(track):
    """Все 5 стемов трека -> {mix, drums, bass, other, vocals} стерео."""
    stems = track.stems                       # (5, N, 2)
    return {
        "mix": stems[0].astype(np.float32),
        "drums": stems[1].astype(np.float32),
        "bass": stems[2].astype(np.float32),
        "other": stems[3].astype(np.float32),
        "vocals": stems[4].astype(np.float32),
    }


def load_song(track):
    """Полная песня: mix + 4 стема как стерео float32 (через API musdb)."""
    return _stems_dict(track)


In [ ]:
class SongCache:
    """LRU-кэш декодированных песен (стерео): полный декод стоит ~2 c,
    повторять его для каждого чанка нельзя."""

    def __init__(self, max_songs=3):
        self.max_songs = max_songs
        self.data = OrderedDict()

    def get(self, track):
        if track.name not in self.data:
            self.data[track.name] = load_song(track)
            while len(self.data) > self.max_songs:
                self.data.popitem(last=False)
        self.data.move_to_end(track.name)
        return self.data[track.name]


class MusicDataset(torch.utils.data.Dataset):
    """Блочная выборка по песням.

    Каждые `chunks_per_song` подряд идущих сэмплов берутся из одного трека,
    поэтому LRU-кэш попадает и каждая песня декодируется один раз за эпоху.
    Аугментация random crop: из полного трека (~7 с) вырезается случайный
    непрерывный отрезок длиной `CHUNK_SAMPLES` (5 с).
    Порядок песен перемешивается при старте каждой эпохи (block 0).
    DataLoader нужно создавать с shuffle=False. При `shuffle_epochs=False`
    чанки детерминированные — для валидации.
    """

    def __init__(self, tracks, chunks_per_song=8, cache=None, shuffle_epochs=True):
        self.tracks = list(tracks)
        self.chunks_per_song = chunks_per_song
        self.cache = cache if cache is not None else SongCache()
        self.shuffle_epochs = shuffle_epochs
        self.order = list(range(len(self.tracks)))

    def __len__(self):
        return len(self.tracks) * self.chunks_per_song

    def __getitem__(self, idx):
        block = idx // self.chunks_per_song
        if self.shuffle_epochs and block == 0:
            random.shuffle(self.order)   # новая эпоха: новый порядок песен
        track = self.tracks[self.order[block % len(self.tracks)]]
        song = self.cache.get(track)

        span = max(0, len(song["mix"]) - CHUNK_SAMPLES)
        if self.shuffle_epochs:
            start = random.randint(0, span)
        else:
            start = (idx * 1009) % (span + 1)
        stop = start + CHUNK_SAMPLES

        stems = torch.as_tensor(
            np.stack([song[k] for k in SOURCE_NAMES])[:, start:stop]
        )                                   # (4, N, 2)
        mix = torch.as_tensor(song["mix"][start:stop])   # (N, 2)
        if mix.numel() < CHUNK_SAMPLES * 2:
            pad = CHUNK_SAMPLES - mix.shape[0]
            mix = F.pad(mix, (0, 0, 0, pad))
            stems = F.pad(stems, (0, 0, 0, pad))
        return mix, stems   # (N, 2) float32, (4, N, 2) float32


mus_train = musdb.DB(root=DB_ROOT, subsets="train", sample_rate=SAMPLE_RATE)
mus_test = musdb.DB(root=DB_ROOT, subsets="test", sample_rate=SAMPLE_RATE)
print("train tracks:", len(mus_train), "| test tracks:", len(mus_test))

train_ds = MusicDataset(mus_train, chunks_per_song=8)
val_ds = MusicDataset(mus_test, chunks_per_song=2, shuffle_epochs=False)
print("train:", len(train_ds), "| val:", len(val_ds))


## 2. Аугментация

Demucs-стайл на волновой форме, до пересборки микса: случайный гейн,
гладкий FIR-эквалайзер, питч-шифт ±2.5 полутона, своп L/R каналов,
mixup стемов между треками батча (chunk swap), stem dropout, crosstalk
(утечки стемов друг в друга), итоговый гейн микса; после пересборки в
микс добавляется белый шум (в стемы — нет). Микс всегда пересобирается
из аугментированных стемов.


In [ ]:
def chunk_swap(stems):
    """Mixup-аугментация Demucs-стиля: каждый элемент батча с вероятностью 0.5
    обменивается случайным подмножеством стемов с другим треком батча.
    Работает с (B, C, 2, N)."""
    B, C, _, _ = stems.shape
    if B < 2:
        return stems
    for i in range(B):
        if random.random() < 0.5:
            j = random.randrange(B - 1)
            j = j if j < i else j + 1
            subset = [s for s in range(C) if random.random() < 0.5]
            if subset:
                g = random.uniform(0.2, 0.8)
                stems[i, subset] = g * stems[j, subset] + (1 - g) * stems[i, subset]
    return stems


def pitch_shift(x, n_semitones):
    """Грубый питч-шифт через ресемплинг (linear interpolate):
    x: (B, C, 2, N) -> (B, C, 2, N) той же длины."""
    r = 2.0 ** (n_semitones / 12.0)
    B, C, _, N = x.shape
    sh = x.reshape(B, C * 2, N)
    n_new = max(int(N * r), 1)
    sh = F.interpolate(sh, size=n_new, mode="linear", align_corners=False)
    sh = F.interpolate(sh, size=N, mode="linear", align_corners=False)
    return sh.reshape(B, C, 2, N)


def augment_batch(mix, stems):
    """Demucs-стайл: случайный гейн + FIR-эквалайзер на каждый стем,
    питч-шифт ±2.5 полутона, своп L/R, mixup между треками, stem dropout,
    crosstalk (утечки стемов), затем микс пересобирается; в микс
    добавляется белый шум (в стемы — нет). mix: (B, N, 2),
    stems: (B, C, N, 2)."""
    stems = stems.permute(0, 1, 3, 2).contiguous()        # (B, C, 2, N)
    B, C, Ch, N = stems.shape
    gains = torch.exp(torch.rand(B, C, 1, 1, device=stems.device) * 0.72 - 0.36)
    stems = stems * gains
    kern = torch.randn(B * C * Ch, 1, 16, device=stems.device)
    kern = torch.cumsum(kern, dim=2)
    kern = kern / (kern.abs().sum(dim=2, keepdim=True) + EPS)
    stems = F.conv1d(stems.view(1, B * C * Ch, N), kern, padding=8,
                     groups=B * C * Ch)
    stems = stems.squeeze(0)[:, :N].view(B, C, Ch, N)

    for b in range(B):
        for c in range(C):
            stems[b:b+1, c:c+1] = pitch_shift(
                stems[b:b+1, c:c+1], random.uniform(-2.5, 2.5)
            )

    if random.random() < 0.5:
        stems = stems.flip(2)          # своп L/R каналов

    stems = chunk_swap(stems)              # mixup между треками батча

    # stem dropout: обнуляем стем целиком (его нет в миксе и таргете)
    drop = torch.rand(B, C, 1, 1, device=stems.device) < AUG_STEM_DROP_P
    stems = stems * (~drop).float()

    # crosstalk: с вероятностью AUG_BLEED_P в стем подмешиваем другой стем
    idx = torch.zeros(B, C, dtype=torch.long, device=stems.device)
    bleed = torch.zeros(B, C, 1, 1, device=stems.device)
    for b in range(B):
        for c in range(C):
            if random.random() < AUG_BLEED_P:
                j = random.randrange(C - 1)
                idx[b, c] = j if j < c else j + 1   # j != c
                bleed[b, c, 0, 0] = random.uniform(*AUG_BLEED_G)
    stems = stems + bleed * stems.gather(
        1, idx.view(B, C, 1, 1).expand(B, C, Ch, N)
    )

    mix = stems.sum(dim=1, keepdim=True)   # (B, 1, Ch, N)

    # белый шум в микс: SNR из AUG_NOISE_SNR, амплитуда от std микса
    if AUG_NOISE_SNR is not None:
        snr = torch.empty(B, 1, 1, 1, device=stems.device).uniform_(*AUG_NOISE_SNR)
        noise_amp = mix.std(dim=-1, keepdim=True) * 10 ** (-snr / 20)
        mix = mix + noise_amp * torch.randn_like(mix)

    g = torch.rand(B, 1, 1, 1, device=stems.device) * 0.45 + 0.8
    mix, stems = mix * g, stems * g
    return mix.squeeze(1).permute(0, 2, 1), stems.permute(0, 1, 3, 2)


## 3. Модель — каскад «2D первая, 1D вторая» (реверсивный гибрид)

В отличие от `separator_pytorch.ipynb` (1D-ветка первая, 2D вторая,
объединение обучаемым гейтом), здесь порядок обратный и связь каскадная:

- **Первая — 2D спектральная ветка** (`STFTBranch`): 2D U-Net по
  log-магнитуде STFT микса (каждый канал стерео независимо, общие веса)
  -> 4 конкурентные маски (softmax по стемам на T-F бине). Стемы
  восстанавливаются маскированием комплексного STFT микса + ISTFT —
  фаза достаётся от микса, маски делят энергию «по-честному»;
- **Вторая — 1D WaveUNet-рефайнер** (`WaveUNetRefine`): вход — все 4 стема
  после ISTFT, свёрнутые в 8 каналов (4 стема × 2 канала), U-Net stride-4
  с GLU, dilated bottleneck, skips; глобальный residual — входные стемы
  с обучаемым масштабом `dummies`: `out += (1 + dummies) * spec_stems`.
  Именно эта ветка может править фазу, которую 2D-ветка не видит
  (она работает по log-магнитуде).

Итого ~38.6M параметров: 2D — ~2.6M, 1D-рефайнер — ~36M.


In [ ]:
class EncBlock(nn.Module):
    """Энкодер-блок: conv1d stride-4 + GLU (каналы x2 до GLU)."""
    def __init__(self, cin, cout, kernel=8, stride=4):
        super().__init__()
        self.conv = nn.Conv1d(cin, cout * 2, kernel, stride=stride,
                             padding=kernel // 2)
        self.glu = nn.GLU(dim=1)

    def forward(self, x):
        return self.glu(self.conv(x))


class Bottleneck(nn.Module):
    """Два dilated-conv слоя с GLU и residual-связью."""
    def __init__(self, channels, kernel=3):
        super().__init__()
        self.conv1 = nn.Conv1d(channels, channels * 2, kernel, padding=1)
        self.glu1 = nn.GLU(dim=1)
        self.conv2 = nn.Conv1d(channels, channels * 2, kernel, padding=2, dilation=2)
        self.glu2 = nn.GLU(dim=1)

    def forward(self, x):
        return x + self.glu2(self.conv2(self.glu1(self.conv1(x))))


class DecBlock(nn.Module):
    """Декодер-блок: ConvTranspose stride-4 + GLU + skip + conv 3x3."""
    def __init__(self, cin, cout, kernel=8, stride=4):
        super().__init__()
        self.deconv = nn.ConvTranspose1d(cin, cout * 2, kernel, stride=stride,
                                         padding=kernel // 2, output_padding=1)
        self.glu = nn.GLU(dim=1)
        self.conv = nn.Conv1d(cout, cout, 3, padding=1)

    def forward(self, x, skip=None):
        x = self.glu(self.deconv(x))
        if skip is not None:
            if x.shape[-1] > skip.shape[-1]:
                x = x[..., :skip.shape[-1]]
            elif x.shape[-1] < skip.shape[-1]:
                x = F.pad(x, (0, skip.shape[-1] - x.shape[-1]))
            x = x + skip
        return self.conv(x)


class WaveUNetRefine(nn.Module):
    """1D waveform U-Net-рефайнер (вторая ветка каскада).

    Вход: (B, S, 2, N) — стемы после ISTFT из 2D-ветки (стерео);
    выход: (B, S, 2, N) — исправленные стемы. Внутри вход сводится
    в 8 каналов (S*2), идёт U-Net stride-4 с GLU, в конце к выходу
    прибавляются входные стемы с обучаемым масштабом `dummies`.
    """

    def __init__(self, n_sources=N_SOURCES, channels=64, levels=5,
                 kernel=8, stride=4):
        super().__init__()
        self.stride = stride
        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()
        cin = n_sources * 2                 # 4 стема x 2 канала
        for i in range(levels):
            cout = channels * (2 ** i)
            self.encoders.append(EncBlock(cin, cout, kernel, stride))
            cin = cout
        for i in range(levels):
            cout = channels * (2 ** max(0, i - 1))
            cin = channels * (2 ** i) if i == levels - 1 else channels * (2 ** max(0, i))
            self.decoders.append(DecBlock(cin, cout, kernel, stride))
        self.bottleneck = Bottleneck(channels * (2 ** (levels - 1)))
        self.head = nn.Conv1d(channels, n_sources * 2, 1)
        self.dummies = nn.Parameter(torch.zeros(n_sources))  # глобальный residual

    def forward(self, x):
        mix0 = x
        B, S, Ch, N = x.shape
        x = x.reshape(B, S * Ch, N)          # (B, S*2, N)
        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
        x = self.bottleneck(x)
        for dec, skip in zip(self.decoders[::-1], skips[-2::-1]):
            x = dec(x, skip)
        x = self.decoders[0](x)
        if x.shape[-1] < N:                  # робастность к неделимым длинам
            x = F.pad(x, (0, N - x.shape[-1]))
        x = self.head(x)[..., :N]            # (B, S*2, N)
        x = x.view(B, S, 2, N)               # (B, S, 2, N)
        return x + (1 + self.dummies)[None, :, None, None] * mix0


class SpecBlock(nn.Module):
    """2D conv-блок: conv3x3 -> GroupNorm -> SiLU -> conv3x3 -> GroupNorm + residual."""
    def __init__(self, cin, cout):
        super().__init__()
        self.conv1 = nn.Conv2d(cin, cout, 3, padding=1)
        self.gn1 = nn.GroupNorm(min(cout, 8), cout)
        self.conv2 = nn.Conv2d(cout, cout, 3, padding=1)
        self.gn2 = nn.GroupNorm(min(cout, 8), cout)
        self.act = nn.SiLU(inplace=True)
        self.shortcut = nn.Conv2d(cin, cout, 1) if cin != cout else nn.Identity()

    def forward(self, x):
        h = self.act(self.gn1(self.conv1(x)))
        h = self.gn2(self.conv2(h))
        return self.act(h + self.shortcut(x))


class BiLSTMBottleneck(nn.Module):
    """BiLSTM по оси времени в bottleneck 2D U-Net."""

    def __init__(self, channels, hidden=32, f_pool=16):
        super().__init__()
        self.hidden = hidden
        self.f_pool = f_pool
        self.proj_in = nn.Conv2d(channels, hidden, 1)
        self.lstm = nn.LSTM(hidden * f_pool, hidden * f_pool // 2, batch_first=True,
                           bidirectional=True)
        self.proj_out = nn.Conv2d(hidden, channels, 1)

    def forward(self, x):
        b, c, f, t = x.shape
        h = self.proj_in(x)                                  # (B, H, F, T)
        h = F.interpolate(h, size=(self.f_pool, t), mode="bilinear",
                          align_corners=False)               # (B, H, P, T)
        h = h.permute(0, 3, 1, 2).reshape(b, t, -1)          # (B, T, H*P)
        h, _ = self.lstm(h)                                  # (B, T, H)
        h = h.reshape(b, t, self.hidden, self.f_pool)
        h = h.permute(0, 2, 3, 1)                            # (B, H, P, T)
        h = F.interpolate(h, size=(f, t), mode="bilinear",
                          align_corners=False)               # (B, H, F, T)
        return self.proj_out(h) + x


class STFTBranch(nn.Module):
    """2D U-Net по log-магнитуде STFT микса -> 4 конкурентные маски
    (softmax по стемам на каждом T-F бине: сумма масок = 1).
    Первая ветка каскада — задаёт маски и стемы с фазой микса."""
    def __init__(self, n_sources=N_SOURCES, base=16, levels=4):
        super().__init__()
        self.enc = nn.ModuleList()
        self.dec = nn.ModuleList()
        for i in range(levels):
            cin = 1 if i == 0 else base * 2 ** (i - 1)
            cout = base * 2 ** i
            self.enc.append(SpecBlock(cin, cout))
        c = base * 2 ** (levels - 1)
        self.bottleneck = BiLSTMBottleneck(c)
        for i in range(levels - 1, -1, -1):
            cout = base * 2 ** i
            cin = c + c if i == levels - 1 else base * 2 ** i * 3
            self.dec.append(SpecBlock(cin, cout))
        self.head = nn.Conv2d(base, n_sources, 1)

    def forward(self, spec):
        skips = []
        x = spec
        for enc in self.enc:
            x = enc(x)
            skips.append(x)
            x = F.avg_pool2d(x, 2)
        x = self.bottleneck(x)
        for dec, skip in zip(self.dec, skips[::-1]):
            x = F.interpolate(x, size=skip.shape[-2:], mode="bilinear",
                              align_corners=False)
            x = torch.cat([x, skip], dim=1)
            x = dec(x)
        return self.head(x).softmax(dim=1)


class ReversedHybridUNet(nn.Module):
    """Каскад «2D первая, 1D вторая» (реверсивный гибрид).

    Вход: (B, 2, N) нормализованный микс; выход: (B, 4, 2, N) стемы.

    - 2D `STFTBranch` строит мягкие маски по log-магнитуде STFT микса
      (каждый канал независимо, общие веса), стемы восстанавливаются
      маскированием комплексного STFT + ISTFT (фаза микса);
    - 1D `WaveUNetRefine` принимает эти стемы и правит waveform
      (в т.ч. фазу), с глобальным residual на входные стемы.
    """
    def __init__(self, n_sources=N_SOURCES, channels=64, levels=5,
                 nfft=4096, hop=1024, spec_base=16, spec_levels=4):
        super().__init__()
        self.spec = STFTBranch(n_sources=n_sources, base=spec_base, levels=spec_levels)
        self.wave = WaveUNetRefine(n_sources=n_sources, channels=channels, levels=levels)
        self.nfft = nfft
        self.hop = hop

    def forward(self, x):
        B = x.shape[0]
        N = x.shape[-1]
        xf = x.float()
        win = torch.hann_window(self.nfft, device=x.device)
        st = torch.stft(xf.reshape(-1, N), self.nfft, self.hop,
                        window=win, return_complex=True)      # (B*2, F, T)
        masks = self.spec(st.abs().log1p().unsqueeze(1))      # (B*2, 4, F, T)
        spec_stems = torch.istft(
            (masks * st.unsqueeze(1)).view(B * 2 * N_SOURCES, *st.shape[-2:]),
            self.nfft, self.hop, window=win, length=N,
        )
        spec_stems = spec_stems.view(B, 2, N_SOURCES, N) \
                                 .permute(0, 2, 1, 3)         # (B, 4, 2, N)
        return self.wave(spec_stems)


def build_model(channels=WAVE_CHANNELS, levels=WAVE_LEVELS):
    model = ReversedHybridUNet(n_sources=N_SOURCES, channels=channels, levels=levels,
                                      nfft=SPEC_NFFT, hop=SPEC_HOP)
    if torch.cuda.device_count() > 1:
        print(f"Используем {torch.cuda.device_count()} GPU!")
        model = nn.DataParallel(model).to(DEVICE)

    return model

model = build_model().to(DEVICE)

if torch.cuda.device_count() > 1:
    print(f"Используем {torch.cuda.device_count()} GPU!")
    model = nn.DataParallel(model).to(DEVICE)

n_params = sum(p.numel() for p in model.parameters())
print(f"params: {n_params:,}")

# --- проверка прохода через сеть ---
with torch.no_grad():
    x0 = torch.randn(1, 2, PAD_LEN, device=DEVICE)
    out = model(x0)
print("input :", tuple(x0.shape))
print("output:", tuple(out.shape))


In [ ]:
# ============================================================
# Вариант рефайнера с BiGRU-bottleneck вместо dilated-conv
# ============================================================
# Идея: encoder/decoder U-Net остаются как в WaveUNetRefine (stride-4,
# GLU, skips), но bottleneck (два dilated-conv) заменён на BiGRU по оси
# времени. На входной частоте 4**LEVELS ниже sr: чанк 6 с @ 32 кГц
# -> ~188 шагов, поэтому BiGRU тут дёшев и даёт глобальный контекст
# на весь чанк (аналог BiLSTMBottleneck в 2D-ветке).


class GRUBottleneck(nn.Module):
    """BiGRU по оси времени в bottleneck U-Net (вместо dilated-conv).

    Вход/выход: (B, C, T) на bottleneck-частоте (T ~ 188 для чанка 6 с).
    Proj-свёртки 1x1 до/после GRU + residual на вход.
    """
    def __init__(self, channels, hidden=512, layers=2):
        super().__init__()
        self.proj_in = nn.Conv1d(channels, hidden, 1)
        self.gru = nn.GRU(hidden, hidden // 2, num_layers=layers,
                         batch_first=True, bidirectional=True)
        self.proj_out = nn.Conv1d(hidden, channels, 1)

    def forward(self, x):
        h = self.proj_in(x).transpose(1, 2)      # (B, T, H)
        h, _ = self.gru(h)                        # (B, T, H)
        h = h.transpose(1, 2)                     # (B, H, T)
        return self.proj_out(h) + x


class WaveRNNRefine(nn.Module):
    """1D waveform-рефайнер с BiGRU-bottleneck (вторая ветка каскада).

    Тот же U-Net stride-4 с GLU и skips, что и WaveUNetRefine, но вместо
    dilated-conv Bottleneck — GRUBottleneck: глобальный контекст по времени
    на bottleneck-частоте, меньше параметров (~3.4M против ~4M dilated).
    Вход: (B, S, 2, N); выход: (B, S, 2, N) с residual на входные стемы
    (обучаемый масштаб `dummies`).
    """
    def __init__(self, n_sources=N_SOURCES, channels=WAVE_CHANNELS, levels=LEVELS,
                 kernel=8, stride=STRIDE, gru_hidden=512, gru_layers=2):
        super().__init__()
        self.stride = stride
        self.encoders = nn.ModuleList()
        self.decoders = nn.ModuleList()
        cin = n_sources * 2                 # 4 стема x 2 канала
        for i in range(levels):
            cout = channels * (2 ** i)
            self.encoders.append(EncBlock(cin, cout, kernel, stride))
            cin = cout
        for i in range(levels):
            cout = channels * (2 ** max(0, i - 1))
            cin = channels * (2 ** i) if i == levels - 1 else channels * (2 ** max(0, i))
            self.decoders.append(DecBlock(cin, cout, kernel, stride))
        self.bottleneck = GRUBottleneck(channels * (2 ** (levels - 1)),
                                        gru_hidden, gru_layers)
        self.head = nn.Conv1d(channels, n_sources * 2, 1)
        self.dummies = nn.Parameter(torch.zeros(n_sources))  # глобальный residual

    def forward(self, x):
        mix0 = x
        B, S, Ch, N = x.shape
        x = x.reshape(B, S * Ch, N)          # (B, S*2, N)
        skips = []
        for enc in self.encoders:
            x = enc(x)
            skips.append(x)
        x = self.bottleneck(x)
        for dec, skip in zip(self.decoders[::-1], skips[-2::-1]):
            x = dec(x, skip)
        x = self.decoders[0](x)
        if x.shape[-1] < N:                  # робастность к неделимым длинам
            x = F.pad(x, (0, N - x.shape[-1]))
        x = self.head(x)[..., :N]            # (B, S*2, N)
        x = x.view(B, S, 2, N)               # (B, S, 2, N)
        return x + (1 + self.dummies)[None, :, None, None] * mix0


class ReversedHybridUNetGRU(nn.Module):
    """Каскад «2D первая, BiGRU-рефайнер вторая».

    То же, что ReversedHybridUNet, но вторая ветка — WaveRNNRefine
    (U-Net с BiGRU-bottleneck) вместо WaveUNetRefine (dilated-conv).
    """
    def __init__(self, n_sources=N_SOURCES, channels=WAVE_CHANNELS, levels=LEVELS,
                 nfft=SPEC_NFFT, hop=SPEC_HOP, spec_base=16, spec_levels=4,
                 gru_hidden=512, gru_layers=2):
        super().__init__()
        self.spec = STFTBranch(n_sources=n_sources, base=spec_base, levels=spec_levels)
        self.wave = WaveRNNRefine(n_sources=n_sources, channels=channels,
                                  levels=levels, gru_hidden=gru_hidden,
                                  gru_layers=gru_layers)
        self.nfft = nfft
        self.hop = hop

    def forward(self, x):
        B = x.shape[0]
        N = x.shape[-1]
        xf = x.float()
        win = torch.hann_window(self.nfft, device=x.device)
        st = torch.stft(xf.reshape(-1, N), self.nfft, self.hop,
                        window=win, return_complex=True)      # (B*2, F, T)
        masks = self.spec(st.abs().log1p().unsqueeze(1))      # (B*2, 4, F, T)
        spec_stems = torch.istft(
            (masks * st.unsqueeze(1)).view(B * 2 * N_SOURCES, *st.shape[-2:]),
            self.nfft, self.hop, window=win, length=N,
        )
        spec_stems = spec_stems.view(B, 2, N_SOURCES, N) \
                                 .permute(0, 2, 1, 3)         # (B, 4, 2, N)
        return self.wave(spec_stems)


def build_model_gru(channels=WAVE_CHANNELS, levels=LEVELS, gru_hidden=512, gru_layers=2):
    model = ReversedHybridUNetGRU(n_sources=N_SOURCES, channels=channels, levels=levels,
                                 nfft=SPEC_NFFT, hop=SPEC_HOP, gru_hidden=gru_hidden,
                                 gru_layers=gru_layers)
    
    if torch.cuda.device_count() > 1:
            print(f"Используем {torch.cuda.device_count()} GPU!")
            model = nn.DataParallel(model).to(DEVICE)
    
    return model 

model_gru = build_model_gru().to(DEVICE)

n_params = sum(p.numel() for p in model_gru.parameters())
print(f"params: {n_params:,}")

# --- проверка прохода через сеть ---
with torch.no_grad():
    x0 = torch.randn(1, 2, PAD_LEN, device=DEVICE)
    out = model_gru(x0)
print("input :", tuple(x0.shape))
print("output:", tuple(out.shape))


## 4. Loss

Идентично `separator_pytorch.ipynb`:

- **L1 на волновой форме** — основной член;
- **мультирезолюционная L1 на log-магнитудах STFT** (по real+imag) —
  спектральная поддержка;
- **SI-SDR на волновой форме** — напрямую оптимизирует итоговую метрику;
- **mixture consistency** (`W_CONSIST * L1(sum(est) - mix)`) — сумма стемов
  обязана совпадать с миксом.


In [ ]:
def si_sdr_torch(est, tgt, eps=1e-8):
    """Дифференцируемый SI-SDR (дБ): (B, C, 2, N) -> скаляр (среднее)."""
    est = est - est.mean(dim=-1, keepdim=True)
    tgt = tgt - tgt.mean(dim=-1, keepdim=True)
    inner = (tgt * est).sum(dim=-1)
    denom = (tgt * tgt).sum(dim=-1) + eps
    alpha = inner / denom
    dist = est - alpha[..., None] * tgt
    num = alpha ** 2 * (tgt * tgt).sum(dim=-1)
    den = (dist * dist).sum(dim=-1) + eps
    return (10 * torch.log10(num / den + eps)).mean()


class WaveLoss(nn.Module):
    """Мультирезолюционная комплексная спектральная L1 (real+imag) + L1 на
    волновой форме + SI-SDR:

    l = w_l1 * L1(wave) + w_spec * L1(spec) + w_sdr * (-SI-SDR)
    """

    def __init__(self, spec_configs=MR_LOSS_CONFIGS, w_l1=1.0, w_spec=1.0, w_sdr=0.5):
        super().__init__()
        self.spec_configs = spec_configs
        self.w_l1 = w_l1
        self.w_spec = w_spec
        self.w_sdr = w_sdr

    def forward(self, est, tgt):
        # est, tgt: (B, C, 2, N) — нормализованные waveform стемы (стерео)
        l_spec = 0.0
        for fl, fs in self.spec_configs:
            e = stft(est, fl, fs)      # комплексный STFT
            t = stft(tgt, fl, fs)
            l_spec += F.l1_loss(e.real, t.real) + F.l1_loss(e.imag, t.imag)
        l_spec /= len(self.spec_configs)
        l_wave = F.l1_loss(est, tgt)
        l_sdr = -si_sdr_torch(est, tgt)
        return self.w_l1 * l_wave + self.w_spec * l_spec + self.w_sdr * l_sdr


# STFT-хелперы (sqrt-hann окно, кэш) — как в model.py / separator_pytorch
_WINDOWS = {}


def get_window(n_fft, device):
    key = (n_fft, str(device))
    if key not in _WINDOWS:
        _WINDOWS[key] = torch.sqrt(
            torch.hann_window(n_fft, periodic=True, device=device)
        )
    return _WINDOWS[key]


def stft(x, fl, fs):
    """Комплексный STFT: (..., N) -> (..., F, T)."""
    shape = x.shape[:-1]
    s = torch.stft(
        x.reshape(-1, x.shape[-1]),
        n_fft=fl, hop_length=fs, win_length=fl,
        window=get_window(fl, x.device), return_complex=True,
    )
    return s.reshape(shape + s.shape[-2:])


loss_fn = WaveLoss().to(DEVICE)
print(loss_fn)


## 5. Сравнение моделей (свип)

Прогоняем несколько конфигураций **1D-рефайнера** — разные `channels`
(ширина) и `levels` (глубина) — по `SWEEP_EPOCHS` эпох каждая (мало эпох =
быстрая ранняя оценка пригодности). Нормализация чанка per-chunk по std
микса, AMP на CUDA, grad clipping, cosine annealing, детерминированный val,
best-чекпойнт по val loss в `checkpoints_pytorch/sweep_21d/`.


In [ ]:
def run_epoch(model, loader, optimizer, loss_fn, device, train=True, scaler=None,
              accum_steps=1):
    model.train(train)
    total, n = 0.0, 0
    optimizer.zero_grad(set_to_none=True)
    with torch.enable_grad() if train else torch.no_grad():
        for step, (mix, stems) in enumerate(loader):
            mix = mix.to(device, non_blocking=True)        # (B, N, 2)
            stems = stems.to(device, non_blocking=True)    # (B, 4, N, 2)
            if train:
                mix, stems = augment_batch(mix, stems)
            # per-chunk нормализация по std микса (по каналам отдельно)
            scale = mix.std(dim=1).clamp_min(1e-4)         # (B, 2)
            mix = F.pad((mix / scale[:, None, :]).permute(0, 2, 1),
                        (0, PAD_LEN - CHUNK_SAMPLES))      # (B, 2, N+pad)
            stems = stems.permute(0, 1, 3, 2) / scale[:, None, :, None]   # (B, 4, 2, N)

            with torch.autocast(device_type=device, enabled=scaler is not None and USE_AMP):
                pred = model(mix)[..., :CHUNK_SAMPLES]
                # mixture consistency: сумма стемов обязана совпадать с миксом
                loss = loss_fn(pred, stems) + W_CONSIST * F.l1_loss(
                    pred.sum(dim=1), mix[..., :CHUNK_SAMPLES]
                )

            if train:
                # градиентное накопление: эффективный батч = B * accum_steps
                if scaler is not None:
                    scaler.scale(loss / accum_steps).backward()
                else:
                    (loss / accum_steps).backward()
                if (step + 1) % accum_steps == 0:
                    if scaler is not None:
                        scaler.unscale_(optimizer)
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
                        scaler.step(optimizer)
                        scaler.update()
                    else:
                        torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
                        optimizer.step()
                    optimizer.zero_grad(set_to_none=True)
            total += float(loss.detach())
            n += 1
    if train and (step + 1) % accum_steps != 0:   # добрасываем хвост накопления
        if scaler is not None:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            scaler.step(optimizer)
            scaler.update()
        else:
            torch.nn.utils.clip_grad_norm_(model.parameters(), 10.0)
            optimizer.step()
        optimizer.zero_grad(set_to_none=True)
    return total / n


In [ ]:
def train_single(model, epochs, lr=SWEEP_LR, accum_steps=2, tag=""):
    """Обучение одной модели `epochs` эпох -> (последний train loss, best val)."""
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=epochs)
    scaler = torch.amp.GradScaler(enabled=USE_AMP)
    best_val, last_tr = float("inf"), float("inf")
    history = {"train": [], "val": []}
    for epoch in range(1, epochs + 1):
        tr = run_epoch(model, train_loader, optimizer, loss_fn, DEVICE,
                       train=True, scaler=scaler, accum_steps=accum_steps)
        va = run_epoch(model, val_loader, optimizer, loss_fn, DEVICE, train=False)
        scheduler.step()
        history["train"].append(tr)
        history["val"].append(va)
        is_best = va < best_val
        if is_best:
            best_val = va
        torch.save(model.state_dict(), f"{SWEEP_DIR}/{tag}_e{epoch:02d}.pt")
        print(f"  epoch {epoch:2d} | train {tr:.4f} | val {va:.4f}"
              + ("  (best)" if is_best else ""))
        last_tr = tr
    history_sweep[tag] = history
    return last_tr, best_val


train_loader = torch.utils.data.DataLoader(
    train_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True
)
val_loader = torch.utils.data.DataLoader(
    val_ds, batch_size=4, shuffle=False, num_workers=2, pin_memory=True
)

os.makedirs(SWEEP_DIR, exist_ok=True)
results_sweep = {}    # (ch, lv) -> {params, train, val}
history_sweep = {}    # "ch{ch}_lv{lv}" -> {train: [...], val: [...]} по эпохам

for ch, lv in SWEEP_CONFIGS:
    print(f"\n=== модель: channels={ch}, levels={lv} ===")
    m = build_model(ch, lv).to(DEVICE)
    n_params = sum(p.numel() for p in m.parameters())
    print(f"params: {n_params:,}")
    tag = f"ch{ch}_lv{lv}"
    tr, va = train_single(m, SWEEP_EPOCHS, tag=tag)
    results_sweep[(ch, lv)] = {"params": n_params, "train": tr, "val": va}
    if isinstance(m, nn.DataParallel):
        m = m.module
    torch.save(m.state_dict(), f"{SWEEP_DIR}/sep_best_{tag}.pt")
    del m
    if DEVICE == "mps":
        torch.mps.empty_cache()

# --- итоговая таблица ---
print("\n" + "=" * 78)
print("СВИП МОДЕЛЕЙ 2D->1D: (channels, levels) | params | train | val loss")
print("=" * 78)
best_key = None
for (ch, lv), r in results_sweep.items():
    best = ""
    if best_key is None or r["val"] < results_sweep[best_key]["val"]:
        best_key = (ch, lv)
        best = " <-- BEST"
    print(f"ch{ch:<3d} lv{lv:<3d} | {r['params']:>10,} | {r['train']:>7.4f} | {r['val']:>7.4f}{best}")
print("=" * 78)
br = results_sweep[best_key]
print(f"Оптимальная модель: channels={best_key[0]}, levels={best_key[1]}, "
      f"params={br['params']:,}, val loss={br['val']:.4f}")

# лучшая модель становится глобальной `model` — её используют инференс и метрика ниже
best_ch, best_lv = best_key
model = build_model(best_ch, best_lv).to(DEVICE)
model.load_state_dict(torch.load(f"{SWEEP_DIR}/sep_best_ch{best_ch}_lv{best_lv}.pt",
                                 map_location=DEVICE))
torch.save(model.state_dict(), "checkpoints_pytorch/sep_best_21d.pt")
print("best model -> `model` (сохранён в checkpoints_pytorch/sep_best_21d.pt)")


In [ ]:
# ============================================================
# Графики обучения для каждого конфига свипа
# ============================================================
import matplotlib.pyplot as plt

best_key = min(results_sweep, key=lambda k: results_sweep[k]["val"])

# 1) Кривые train/val loss по эпохам для каждой модели
n = len(results_sweep)
cols, rows = 4, (n + 3) // 4
fig, axes = plt.subplots(rows, cols, figsize=(4.4 * cols, 3.6 * rows),
                         squeeze=False)
axes = axes.ravel()
for k, ((ch, lv), r) in enumerate(results_sweep.items()):
    hist = history_sweep[f"ch{ch}_lv{lv}"]
    ax = axes[k]
    ep = list(range(1, len(hist["train"]) + 1))
    ax.plot(ep, hist["train"], "o-", label="train")
    ax.plot(ep, hist["val"], "s-", label="val")
    is_best = (ch, lv) == best_key
    ax.set_title(f"ch={ch}, lv={lv} | {r['params'] / 1e6:.1f}M"
                 + ("  (BEST)" if is_best else ""), fontsize=10)
    ax.set_xlabel("epoch")
    ax.set_ylabel("loss")
    ax.grid(alpha=0.3)
    ax.legend(fontsize=8)
    ax.set_xticks(ep)
for k in range(n, len(axes)):
    axes[k].axis("off")
fig.suptitle("Обучение каждой модели свипа 2D->1D: train/val loss по эпохам",
             fontsize=13)
fig.tight_layout(rect=[0, 0, 1, 0.96])
plt.show()

# 2) Val loss всех конфигов на одном графике
fig, ax = plt.subplots(figsize=(10, 5))
for (ch, lv), r in results_sweep.items():
    hist = history_sweep[f"ch{ch}_lv{lv}"]
    ax.plot(range(1, len(hist["val"]) + 1), hist["val"], "o-",
            label=f"ch={ch}, lv={lv} ({r['params'] / 1e6:.1f}M)")
ax.set_xlabel("epoch")
ax.set_ylabel("val loss")
ax.set_title("Val loss всех конфигов (меньше = лучше)")
ax.grid(alpha=0.3)
ax.legend(fontsize=9)
plt.show()

# 3) Сравнение best val loss по конфигам (столбцы)
fig, ax = plt.subplots(figsize=(10, 4))
keys = list(results_sweep.keys())
labels = [f"ch{ch}/lv{lv}\n{r['params'] / 1e6:.1f}M"
          for (ch, lv), r in results_sweep.items()]
vals = [results_sweep[k]["val"] for k in keys]
colors = ["#2e9e5b" if k == best_key else "#4c78a8" for k in keys]
ax.bar(labels, vals, color=colors)
ax.set_ylabel("best val loss")
ax.set_title("Сравнение моделей по best val loss (меньше = лучше)")
ax.grid(axis="y", alpha=0.3)
for i, v in enumerate(vals):
    ax.text(i, v, f"{v:.4f}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()


## 6. Инференс

Песню режем на перекрывающиеся чанки (50% overlap, Hann-кросфейд),
предсказываем стемы каскадом (2D маски -> ISTFT -> 1D-рефайнер),
нормализация per-chunk как в обучении; опционально Wiener soft-mask
пост-процессинг по всему треку.


In [ ]:
def wiener_postprocess(stems, mix, gamma=2.0, fl=4096, fs=1024,
                        seg=30 * SAMPLE_RATE):
    """Wiener soft-mask пост-обработка: маски m_i = |S_i|^gamma / sum_j |S_j|^gamma,
    итог = ISTFT(m_i * STFT(mix)) с фазой микса. Считается сегментами.
    stems: (N, 4, 2), mix: (N, 2) -> (N, 4, 2) float32."""
    out = np.empty_like(stems)
    for start in range(0, len(mix), seg):
        e = stems[start:start + seg]
        m = mix[start:start + seg]
        s = stft(torch.from_numpy(np.ascontiguousarray(e.transpose((1, 2, 0)))).to(DEVICE),
                 fl, fs)                            # (4, 2, F, T)
        mst = stft(torch.from_numpy(np.ascontiguousarray(m.transpose((1, 0)))).to(DEVICE),
                   fl, fs)                          # (2, F, T)
        mag = s.abs().pow(gamma)                     # (4, 2, F, T)
        masks = mag / (mag.sum(dim=0, keepdim=True) + EPS)  # softmax по стемам
        est = (masks * mst.unsqueeze(0)).reshape(-1, *mst.shape[-2:])  # (4*2, F, T)
        r = torch.istft(est, fl, fs, window=get_window(fl, DEVICE), length=len(m))
        out[start:start + seg] = r.reshape(4, 2, len(m)).permute(2, 0, 1).cpu().numpy()
    return out


def separate_track_all(model, mix, device):
    """Полный трек -> все 4 стема (np.float32, (N, 4, 2)): перекрывающиеся
    чанки с Hann-кросфейдом + mixture-consistency проекция в каждом чанке
    и финальный Wiener soft-mask пост-процессинг по всему треку."""
    model.eval()
    n = len(mix)
    hop = CHUNK_SAMPLES // 2

    out = np.zeros((n, 4, 2), dtype=np.float32)
    weights = np.zeros((n, 2), dtype=np.float32)

    with torch.no_grad():
        for st in range(0, max(1, n - CHUNK_SAMPLES + 1), hop):
            chunk = mix[st:st + CHUNK_SAMPLES]            # (N', 2)
            if len(chunk) < CHUNK_SAMPLES:
                chunk = np.pad(chunk, ((0, CHUNK_SAMPLES - len(chunk)), (0, 0)))
            x = torch.from_numpy(chunk).to(device)        # (N, 2)
            scale = x.std(dim=0).clamp_min(1e-4)          # (2,)
            xp = F.pad((x / scale).t().unsqueeze(0),
                       (0, PAD_LEN - CHUNK_SAMPLES))      # (1, 2, N+pad)
            est = model(xp)[:, :, :, :CHUNK_SAMPLES] * scale[:, None]  # (1, 4, 2, N)
            est = est.squeeze(0).cpu().numpy().transpose(2, 0, 1)     # (N, 4, 2)

            # mixture consistency: остаток микса -> стемам по энергии
            residual = chunk - est.sum(1)                  # (N, 2)
            energy = (est ** 2).sum(axis=(0, 2), keepdims=True) + 1e-8  # (1, 4, 1)
            est = est + residual[:, None] * (energy / energy.sum())

            w = np.hanning(CHUNK_SAMPLES).astype(np.float32)

            stop = min(st + CHUNK_SAMPLES, n)
            out[st:stop] += est[:stop - st] * w[:stop - st, None, None]
            weights[st:stop] += w[:stop - st, None]

    ok = weights[:, 0] > 0
    out[ok] /= weights[ok, None]
    return wiener_postprocess(out, mix)


def separate_track(model, mix, device, source_idx=0):
    """Полный трек -> стерео стем (np.float32, (N, 2)),
    перекрывающиеся чанки с кросфейдом."""
    model.eval()
    n = len(mix)
    hop = CHUNK_SAMPLES // 2

    out = np.zeros((n, 2), dtype=np.float32)
    weights = np.zeros(n, dtype=np.float32)

    with torch.no_grad():
        for st in range(0, max(1, n - CHUNK_SAMPLES + 1), hop):
            chunk = mix[st:st + CHUNK_SAMPLES]            # (N', 2)
            if len(chunk) < CHUNK_SAMPLES:
                chunk = np.pad(chunk, ((0, CHUNK_SAMPLES - len(chunk)), (0, 0)))
            x = torch.from_numpy(chunk).to(device)        # (N, 2)
            scale = x.std(dim=0).clamp_min(1e-4)          # (2,)
            xp = F.pad((x / scale).t().unsqueeze(0),
                       (0, PAD_LEN - CHUNK_SAMPLES))      # (1, 2, N+pad)
            est = model(xp)[0, source_idx, :, :CHUNK_SAMPLES] * scale[:, None]
            est = est.t().cpu().numpy()                   # (N, 2)

            w = np.hanning(CHUNK_SAMPLES).astype(np.float32)

            stop = min(st + CHUNK_SAMPLES, n)
            out[st:stop] += est[:stop - st] * w[:stop - st, None]
            weights[st:stop] += w[:stop - st]

    ok = weights > 0
    out[ok] /= weights[ok, None]
    return out


def separate_file(model, track, device, source_idx=3):
    """Стем из musdb.Track (0 drums, 1 bass, 2 other, 3 vocals)."""
    song = load_song(track)
    stem = separate_track(model, song["mix"], device, source_idx=source_idx)
    return stem, song[SOURCE_NAMES[source_idx]], song["mix"]


In [ ]:
# Демо: отделить вокал из первого трека теста
test_track = mus_test[0]

est_vocals, ref_vocals, mix = separate_file(model, test_track, DEVICE, source_idx=3)

def write_44k(path, data):
    """Запись wav в 44.1 кГц (ресемпл из SAMPLE_RATE через ffmpeg)."""
    import stempeg
    stempeg.write_audio(path, data, sample_rate=SAMPLE_RATE,
                        output_sample_rate=44100)


os.makedirs("output", exist_ok=True)
write_44k("output/mix.wav", mix)
write_44k("output/vocals_ref.wav", ref_vocals)
write_44k("output/vocals_pred.wav", est_vocals)

print("wrote output/*.wav |", test_track.name)


## 7. Метрика — SI-SDR

Оценка качества отделённого стема на тестовых треках (SDR в дБ, чем больше — тем лучше).


In [ ]:
def si_sdr(estimate, reference):
    """SI-SDR в дБ: (N,) или (N, 2) -> скаляр (среднее по каналам)."""
    eps = 1e-8
    if estimate.ndim == 1:
        estimate = estimate[:, None]
        reference = reference[:, None]
    estimate = estimate - estimate.mean(axis=0, keepdims=True)
    reference = reference - reference.mean(axis=0, keepdims=True)
    alpha = np.sum(reference * estimate, axis=0) / (
        np.sum(reference * reference, axis=0) + eps
    )
    distortion = estimate - alpha * reference
    sdr = 10 * np.log10(
        alpha ** 2 * np.sum(reference ** 2, axis=0)
        / (np.sum(distortion ** 2, axis=0) + eps)
    )
    return float(sdr.mean())


def evaluate(model, tracks, max_songs=3, source_idx=3):
    sdr_list = []
    for track in tracks[:max_songs]:
        est, ref, _ = separate_file(model, track, DEVICE, source_idx=source_idx)
        n = min(len(est), len(ref))
        sdr_list.append(si_sdr(est[:n], ref[:n]))
        print(f"{track.name[:40]:42s} SI-SDR = {sdr_list[-1]:.1f} dB")
    return np.mean(sdr_list), sdr_list


print(f"Оценка лучшей модели из свипа 2D->1D: channels={best_ch}, levels={best_lv}, "
      f"val loss={results_sweep[(best_ch, best_lv)]['val']:.4f}")
mean_sdr, per_song = evaluate(model, mus_test, max_songs=3, source_idx=3)
print(f"\nmean SI-SDR (vocals): {mean_sdr:.1f} dB")
